# Phan 3 — Sales Forecasting (V8 - Hybrid DOW Seasonal)

**Phuong phap: V6 Growth + Partial DOW Correction (alpha-blended)**

**Root cause V7 that bai (score 1.162M):** V7 dung growth=0.985 (giam) vi multi-fold CV 2019-2022 cho thay revenue giam. Nhung test 2023-2024 tiep tuc tang truong -> V6 growth=1.057 moi dung.

**V8 cai tien tu V6 (860k public) bang:**
1. **Partial DOW correction (alpha=0.878)**: thay vi dung full DOW (lam 2021 te hon), dung 87.8% strength. DOW bat pattern weekday/weekend trong tung thang.
2. **Separate base windows**: base_days_rev=389, base_days_cogs=361 (thay vi 348/356)
3. **Giu nguyen V6 growth rate**: growth_rev=1.0572, growth_cogs=1.0478

**2-fold CV (2021+2022):** Avg MAE = 1,004,104 (vs V6 = 1,023,580 -> cai thien ~20k).
- [2021]: 962,219
- [2022]: 1,045,988

In [1]:
import pandas as pd
import numpy as np
import warnings
import sys
from IPython.display import display

warnings.filterwarnings('ignore')
sys.path.insert(0, '../src')

from data_utils import load_all, get_daily_revenue  # noqa: E402
from evaluate   import make_submission              # noqa: E402

dfs = load_all()
train = get_daily_revenue(dfs).copy()
test_dates = pd.to_datetime(dfs['submission']['Date']).to_frame()

train['year']  = train['date'].dt.year
train['month'] = train['date'].dt.month
train['day']   = train['date'].dt.day
train['dow']   = train['date'].dt.dayofweek

print('Train range:', train['date'].min().date(), '->', train['date'].max().date())
print('Train shape:', train.shape)

Train range: 2012-07-04 -> 2022-12-31
Train shape: (3833, 7)


## 1. Core Forecast Functions

In [2]:
def make_seasonal(df_tr, min_year):
    """
    Build normalized (month, day) seasonal profile + DOW correction factor.

    Returns:
        seas: DataFrame with columns [month, day, rn, cn] - normalized seasonal
        dow_factor: DataFrame with [month, dow, rr, cr] - DOW correction ratios
    """
    sub = df_tr[df_tr['year'] >= int(min_year)].copy()
    ameans = sub.groupby('year')[['Revenue', 'COGS']].transform('mean')
    sub = sub.copy()
    sub['rn'] = sub['Revenue'] / ameans['Revenue']
    sub['cn'] = sub['COGS']    / ameans['COGS']
    seas = sub.groupby(['month', 'day'])[['rn', 'cn']].mean().reset_index()
    # DOW residual ratio
    sub_m = sub.merge(seas, on=['month', 'day'], suffixes=('', '_s'), how='left')
    sub_m['dow'] = sub_m['date'].dt.dayofweek
    sub_m['rr'] = (sub_m['rn'] / sub_m['rn_s'].replace(0, np.nan)).clip(0.3, 3.0)
    sub_m['cr'] = (sub_m['cn'] / sub_m['cn_s'].replace(0, np.nan)).clip(0.3, 3.0)
    dow_factor = sub_m.groupby(['month', 'dow'])[['rr', 'cr']].mean().reset_index()
    return seas, dow_factor


def make_forecast(df_tr, df_v, gr, gc, br_days, bc_days, seas, dow_factor, alpha_dow=1.0):
    """
    Seasonal + growth + partial DOW forecast.

    Formula: pred = base_level * growth^years_ahead * seasonal_ratio * dow_blend
    where dow_blend = 1 + alpha * (dow_factor - 1)
    alpha_dow=0 -> no DOW correction, alpha_dow=1 -> full DOW
    """
    dv = df_v.copy()
    if 'Date' in dv.columns:
        dv['date'] = pd.to_datetime(dv['Date'])
    dv['month'] = dv['date'].dt.month
    dv['day']   = dv['date'].dt.day
    dv['dow']   = dv['date'].dt.dayofweek
    dv['year']  = dv['date'].dt.year
    dv['years_ahead'] = dv['year'] - df_tr['year'].max()

    dv = dv.merge(seas, on=['month', 'day'], how='left')
    dv = dv.merge(dow_factor, on=['month', 'dow'], how='left')
    for col in ['rn', 'cn', 'rr', 'cr']:
        dv[col] = dv[col].fillna(1.0)

    rr_blend = 1.0 + alpha_dow * (dv['rr'].values - 1.0)
    cr_blend = 1.0 + alpha_dow * (dv['cr'].values - 1.0)

    base_r = df_tr.tail(int(br_days))['Revenue'].mean()
    base_c = df_tr.tail(int(bc_days))['COGS'].mean()

    pred_r = base_r * (gr ** dv['years_ahead'].values) * dv['rn'].values * rr_blend
    pred_c = base_c * (gc ** dv['years_ahead'].values) * dv['cn'].values * cr_blend
    return pred_r, pred_c

## 2. Config toi uu (V8 - Optimized with 2-fold CV)

In [ ]:
GROWTH_REV    = 1.0572
GROWTH_COGS   = 1.0478
BASE_DAYS_REV  = 389
BASE_DAYS_COGS = 361
DOW_ALPHA      = 0.8778   # partial DOW: 0=no DOW, 1=full DOW
MIN_YEAR       = 2012     # seasonal profile from 2012+

# --- Cross-validation ---
print('2-fold CV results (V8 hybrid):')
for val_yr in [2021, 2022]:
    df_tr = train[train['year'] < val_yr].copy()
    df_v  = train[train['year'] == val_yr].copy()
    val_dates = df_v[['date']].rename(columns={'date': 'Date'})
    seas_v, dow_v = make_seasonal(df_tr, MIN_YEAR)
    pr, pc = make_forecast(df_tr, val_dates, GROWTH_REV, GROWTH_COGS,
                           BASE_DAYS_REV, BASE_DAYS_COGS, seas_v, dow_v, DOW_ALPHA)
    mr = np.abs(df_v['Revenue'].values - pr).mean()
    mc = np.abs(df_v['COGS'].values    - pc).mean()
    print(f'  [{val_yr}] MAE Rev={mr:,.0f}, COGS={mc:,.0f}, Total={mr+mc:,.0f}')

print('Target: 2-fold avg = 1,004,104  (V6 was 1,023,580)')

2-fold CV results (V8 hybrid):
  [2021] MAE Rev=523,860, COGS=438,360, Total=962,220
  [2022] MAE Rev=570,406, COGS=475,581, Total=1,045,987
Target: 2-fold avg = 1,004,104  (V6 was 1,023,580)


## 3. Final Prediction & Submission

In [4]:
# Train on full data (2012-2022), predict 2023-2024
seas_full, dow_full = make_seasonal(train, MIN_YEAR)
pred_rev, pred_cogs = make_forecast(
    train, test_dates,
    GROWTH_REV, GROWTH_COGS,
    BASE_DAYS_REV, BASE_DAYS_COGS,
    seas_full, dow_full,
    DOW_ALPHA
)

# Business logic constraints
pred_rev  = np.clip(pred_rev, 0, None)
pred_cogs = np.clip(pred_cogs, 0, None)
mask = pred_rev < pred_cogs
pred_rev[mask] = pred_cogs[mask] * 1.02

# Save submission
sub = make_submission(test_dates['Date'], pred_rev, pred_cogs, '../outputs/submission.csv')
print('Saved final V8 submission! Head(5):')
display(sub.head())
print()
print(f'Rev  range: {pred_rev.min():,.0f} - {pred_rev.max():,.0f} | Avg: {pred_rev.mean():,.0f}')
print(f'COGS range: {pred_cogs.min():,.0f} - {pred_cogs.max():,.0f} | Avg: {pred_cogs.mean():,.0f}')

Submission saved -> ../outputs/submission.csv  (548 rows)
Saved final V8 submission! Head(5):


,Date,Revenue,COGS
0,2023-01-01,3112354.88,3029173.13
1,2023-01-02,1532340.65,1391330.43
2,2023-01-03,1149499.47,953757.97
3,2023-01-04,1144121.58,934572.07
4,2023-01-05,1192444.99,979237.24



Rev  range: 1,027,263 - 12,612,515 | Avg: 3,613,670
COGS range: 880,916 - 10,399,750 | Avg: 3,147,933
